# FinSafe 08 — CPU final review and artifact publication
Restores completed GPU research, validates all 144 assistant-reviewed annotations, reruns AuditKIT and publishes explicit artifact folders. No retraining or GPU required. Uses HF_TOKEN and HF_WRITE (or HF_WRITE_TOKEN). Source pinned to 62716f86125d83d5bd95a147047a92d056bbcbb0. Final organizer submission is separate.


In [ ]:
import os,sys,time,datetime,subprocess,json,zipfile,shutil
from pathlib import Path
started=time.monotonic()
print(datetime.datetime.now(datetime.timezone.utc).isoformat(),'START CPU review finalization and publication',flush=True)
try:
    project=Path('/kaggle/working/finsafe')
    pin='62716f86125d83d5bd95a147047a92d056bbcbb0'
    if not project.exists():subprocess.run(['git','clone','https://github.com/dixitkeshav/finsafe.git',str(project)],check=True)
    subprocess.run(['git','-C',str(project),'fetch','origin',pin],check=True)
    subprocess.run(['git','-C',str(project),'checkout','--detach',pin],check=True)
    sys.path.insert(0,str(project/'src'))
    from finsafe.progress import Progress,atomic_json
    progress=Progress(project/'runs/publication-resume/logs','publication-resume')
    subprocess.run([sys.executable,'-m','pip','install','-q','uv','huggingface_hub==1.33.0'],check=True)
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import HfApi,hf_hub_download
    secrets=UserSecretsClient()
    read_token=secrets.get_secret('HF_TOKEN')
    try:write_token=secrets.get_secret('HF_WRITE_TOKEN')
    except Exception:write_token=secrets.get_secret('HF_WRITE')
    env=os.environ.copy();env.update(HF_TOKEN=read_token,HF_WRITE_TOKEN=write_token,MPLBACKEND='Agg',PYTHONUNBUFFERED='1',UV_CACHE_DIR='/kaggle/temp/finsafe-publish-uv',UV_PYTHON_INSTALL_DIR='/kaggle/temp/finsafe-publish-python')
    api=HfApi(token=write_token)
    with progress.stage('01-restore-completed-gpu-experiment'):
        archive=hf_hub_download('Finsafe-team/finsafe-recovery','latest/submission-recovery.zip',repo_type='dataset',token=write_token)
        with zipfile.ZipFile(archive) as z:
            if z.testzip():raise RuntimeError('Recovery CRC failure')
            for member in z.infolist():
                p=Path(member.filename)
                if p.is_absolute() or '..' in p.parts or not p.parts or p.parts[0] not in {'data','runs','checkpoints'} or ((member.external_attr>>16)&0o170000)==0o120000:raise RuntimeError('Unsafe recovery member')
            z.extractall(project)
        run=project/'runs/submission-v1'
        if not (run/'conditions.json').exists():raise RuntimeError('Missing completed GPU conditions')
    with progress.stage('02-CPU-environment'):
        py=str(project/'.venv-publish/bin/python');uv=[sys.executable,'-m','uv']
        if not Path(py).exists():progress.run(uv+['venv','--python','3.12',str(project/'.venv-publish')],cwd=project,env=env)
        progress.run(uv+['pip','install','--python',py,'-e','.','huggingface-hub==1.33.0'],cwd=project,env=env)
        progress.run(uv+['pip','check','--python',py],cwd=project,env=env)
    with progress.stage('03-validate-reviewed-labels-and-AuditKIT'):
        progress.run([py,'-u','scripts/finalize_review.py','--run',str(run)],cwd=project,env=env)
        protocol=json.loads((run/'protocol.json').read_text());protocol['publication_source_revision']=pin
        atomic_json(run/'protocol.json',protocol)
        failure=run/'failed.json'
        if failure.exists():
            (run/'history').mkdir(exist_ok=True);failure.replace(run/'history/gpu-publication-failure.json')
        atomic_json(run/'completed.json',{'status':'complete','training_and_generations':'Kaggle Version 12','review':'all 144 condition-hidden responses read by Codex assistant; no independent human review','publication_resumed_on_CPU':True})
    with progress.stage('04-publication'):
        progress.run([py,'-u','scripts/publish_submission.py','--run',str(run)],cwd=project,env=env)
        links=json.loads((run/'public-links.json').read_text())
        for kind in ['model','dataset']:
            slug=links[kind+'_collection'].split('/collections/')[1]
            api.update_collection_metadata(slug,description='SAK_GURU Lexsi hackathon completed preliminary FinSafe experiment; see measured report and AI-review limitations')
        for name in ['completed.json','compute.json','public-links.json']:
            api.upload_file(path_or_fileobj=str(run/name),path_in_repo=name,repo_id='Finsafe-team/finsafe-runs',repo_type='dataset')
        print('PUBLIC LINKS',json.dumps(links),flush=True)
    with progress.stage('05-retain-portable-output'):
        shutil.rmtree(project/'.venv-publish')
    progress.log('DONE CPU publication. Organizer form still needs final verification and submission.')
except BaseException as error:
    print('FAILED CPU publication',type(error).__name__,'elapsed',round(time.monotonic()-started,1),flush=True)
    raise
else:print('DONE notebook',round(time.monotonic()-started,1),'seconds',flush=True)
